# Training progress, interactive

Widgets for following and comparing runs. The widget code lives in `chl_cnn/progress_ui.py`; the figures come from `chl_cnn/watch.py`, `viz.py`, `attn_viz.py` and `tracer_viz.py`. Needs a live Jupyter / VS Code front end (ipywidgets). Run the set-up cell first, then any section on its own.

Runs are found under `models/<category>/<run>` (`chl_cnn/runs.py`), and optionally under `models/history/<category>/<run>` (archived runs: config, history and metrics only). A run gets its `config.json` only when it finishes; until then its settings are read from the live trainer process or from its log header in `~/mind_the_chl_gap/logs/`. Status in legends: *running* (a trainer process is training it), *unfinished* (no `metrics.json` and no live process), or done.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
%matplotlib inline
import matplotlib.pyplot as plt

plt.ioff()  # figures go only where display() puts them (inside the Output widgets)
from chl_cnn import progress_ui as UI

## Training curves

Val hidden-pixel MAE per epoch (left) and the physics residual of PINN runs (right). Pick categories, then runs (Ctrl/Cmd-click for several); the list starts with the 6 most recently updated runs. **include archived** adds `models/history/` runs (labelled `history/...`). **hide smoke / leak / eval runs** drops runs whose names contain `smoke`, `leak`, `eval`, `regcheck` or `_maps`. **plot every listed run** ignores the highlighted selection. The name glob applies within the categories, e.g. `io3_*` or `*lam0p3`. **Refresh** re-lists the runs and re-reads `history.json` (running jobs rewrite it each epoch).

Static version: `python -m chl_cnn.watch --cat pinn/t3d --glob 'io4_*' --every 60 --out models/progress_t3d.png` (add `--archived` for `models/history/`).

In [ ]:
UI.val_curves()

## Output visualizer: same 32 validation days for any finished run

Each finished run stores its predictions on every val/test day (`preds.npz`). Works for every store (IO, Black Sea 2 km and 4 km); the store, crop, split and donor settings come from the run's config.json. The 32 days are the run's val days with saved predictions, evenly spaced over the observed-fraction range, so runs on the same store and `--val-stride` show the same days. Top row: visible input, prediction, truth, hidden-pixel error. Bottom row: input at t−1 and t+1 as the model sees it, SST, and the error histogram. A day outside the model's input window (e.g. t+1 for the nowcast) is left blank. Loading a store takes a few seconds the first time (about 30 s for the 2 km Black Sea).

The run list holds runs of the chosen category with `preds.npz` and `config.json`, the most recently updated one selected first. Context panels pick t±1 by calendar day, as the trainer does.

In [ ]:
UI.days()

## Attention models (`--t3d-stem attn / patch / tokdec`)

Runs in `models/pinn/t3d` with an attention stem: per-pixel temporal attention (`attn`, `chl_cnn/tattn.py`, e.g. `io2_tattn8`), patch tokens (`patch`, `chl_cnn/ptok.py`; `io3_ptok4` arm A with own-pixel values, `io3_ptok4_lavg` arm B with local averages) and the token decoder (`tokdec`, `chl_cnn/tokdec.py`; `io4_tokdec`, `io4_tokdec_blend`). Pick attention runs and reference runs (Conv3d stem, e.g. `t3d_dil_c4_60ep_gdl100`, `io2_seed1`, `io2_seed2`); all three panels below follow the attention selection. Both lists cover runs still training (settings read from the live process) and are limited to one **store**; **hide smoke / leak / eval runs** works as above. Plot code: `chl_cnn/attn_viz.py`.

**(a) Training curves.** Val hidden-pixel MAE of the attention runs (solid) and references (dashed). The x-axis toggle switches between epoch, optimiser steps (ceil(5,540 training days / batch) per epoch, batch from config.json) and training days seen (5,540 per epoch). The legend gives the stem, batch, median s/epoch (history.json `epoch_s`), epochs logged and best val MAE. **Refresh** re-reads history.json (running jobs rewrite it each epoch) and the run list.

In [ ]:
att = UI.Attention(store="io")
att.curves()

**(b) Pooling weight by day offset.** For each selected attention run with a weight cache: the mean weight per val hidden pixel on each window day (−8..+8; `t` = day t), split by whether that day is visible at the pixel in the model's input (donor-masked), and for patch models (and the `tokdec_blend` blend weights) the "no day" slot; the bars of a panel sum to 1. Hidden pixels = observed on day t and not visible in day t's input, pooled over the cached val days. Weight sets: `attn` pooling weights (mean over heads); `patch` per-pixel day weights; `tokdec` patch pooling weights (mean over heads, each pixel gets its patch's weights) and, with `--tokdec-blend`, the blend day weights.

The caches (`analyses/attn_viz/cache/<run>.npz`, 12 val days, every 122nd val day, about 10 MB) come from the trainer's own eval-only path with the stem recorders on (`analyses/attn_viz/build_cache.py`). A run without a cache prints the command that builds it: on CPU (CUDA hidden), about 1–2 min per run; with `--device cuda` it runs under `flock ~/mind_the_chl_gap/logs/gpu.lock`. A cache older than the run's `model.pt` is flagged. **Redraw** re-reads the caches.

In [ ]:
att.offsets()

**(c) Day-weight maps.** One selected attention run (with a cache) and one of its cached val days: the model's weight on each chosen day offset (columns) next to that day's visible mask in the model's input, land blank, north up (same orientation as the panels above). Patch models also show the two logit terms of the day weight: the patch score (interpolated between patch centres) and the per-pixel term, and for `lavg` the local confidence conv(m). `tokdec` shows the patch pooling weights (one value per 4 × 4 px patch) and, with `--tokdec-blend`, the blend day weights. **offsets** picks the columns (Ctrl/Cmd-click for several); **day t** adds day t for stems that weight it (`attn`, `tokdec` pooling); **no day** adds the "no day" weight of patch models. Weight colour scales run from 0 to the 99.5th percentile over the shown columns.

In [ ]:
att.maps()

## Tracer runs (models/tracer, `train_tracer.py`)

Learned Lagrangian tracer: validation MAE per forecast lead by epoch (model on the common pixel set, baselines as horizontal lines once `metrics.json` exists at the end of a run), MAE by lead at the best epoch, learned factor statistics (a, b, wind c and theta when the run has them), model vs reanalysis speed, and maps of a(x, season) for 4 days of year (from `factor_maps.npz`, else computed on CPU from `model.pt`); for step-2 `--source` runs the basin-mean lead-1 source by epoch and season-mean source maps (`source_maps.npz`). Panels whose input files are missing print a note and are skipped. Plot code: `chl_cnn/tracer_viz.py`.

Only runs in `train_tracer.py` format are listed (the `trb_*` / `trs_*` runs come from other trainers). Nothing is selected at first; the factor maps of runs without `factor_maps.npz` are computed on CPU and take a while.

In [ ]:
UI.tracer()